<div style="background:linear-gradient(135deg,#512E5F 0%,#7D3C98 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#D7BDE2;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 4 · CLASE 11</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: Retener PCs e Interpretación</h1>
  <p style="color:#D7BDE2;font-size:16px;margin:0 0 24px 0;font-style:italic;">Kaiser · Codo · Loadings · Reconstrucción · Pipeline completo</p>
  <hr style="border-color:#A569BD;margin:0 0 20px 0;">
  <p style="color:#F5EEF8;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; 🟢 Básico · 🟡 Intermedio · 🔴 Avanzado</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
np.set_printoptions(precision=4,suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 🟢 Ejercicio 1 — Kaiser y varianza acumulada

1. Ajustar PCA a datos estandarizados
2. Aplicar criterio Kaiser (λ > 1)
3. Aplicar criterio 80% varianza acumulada
4. ¿Coinciden los dos criterios?
5. ¿Cuántos features originales tiene el dataset? ¿Cuántos PCs sugiere retener?

In [ ]:
np.random.seed(SEED)
n_e1, p_e1 = 300, 10
# Señal real en las primeras 4 dimensiones
X_e1 = (np.random.randn(n_e1, 4) @ np.random.randn(4, p_e1)
        + np.random.randn(n_e1, p_e1) * 0.4)
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED); n_e1,p_e1=300,10
X_e1=np.random.randn(n_e1,4)@np.random.randn(4,p_e1)+np.random.randn(n_e1,p_e1)*0.4
pca1=PCA(); pca1.fit(StandardScaler().fit_transform(X_e1))
ev1=pca1.explained_variance_; ev_r1=pca1.explained_variance_ratio_; ev_c1=np.cumsum(ev_r1)
k_kaiser1=int(np.sum(ev1>1)); k_80_1=np.argmax(ev_c1>=0.80)+1
print(f'Criterio Kaiser (λ>1):  k={k_kaiser1}')
print(f'Criterio 80% varianza:  k={k_80_1}')
print(f'Coinciden: {k_kaiser1==k_80_1}')
print(f'\nVarianza acumulada en k={k_kaiser1}: {ev_c1[k_kaiser1-1]:.2%}')
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(range(1,p_e1+1),ev_r1*100,color='#7D3C98',alpha=0.85,edgecolor='white')
ax.axvline(k_kaiser1,color='#E74C3C',lw=2,linestyle='--',label=f'Kaiser k={k_kaiser1}')
ax.axvline(k_80_1,color='#F39C12',lw=2,linestyle='-.',label=f'80% var k={k_80_1}')
ax.set(xlabel='PC',ylabel='% Varianza',title='Criterios de selección')
ax.legend(); ax.grid(True,alpha=0.25); plt.tight_layout(); plt.show()

---
## 🟡 Ejercicio 2 — Interpretación de loadings

1. Ajustar PCA con 3 componentes
2. Construir el heatmap de loadings
3. Identificar qué 2 features tienen mayor loading en PC1
4. Identificar qué features están correlacionadas entre sí (loadings similares)
5. Proponer un nombre para cada PC basado en los loadings

In [ ]:
np.random.seed(SEED)
feats_e2=['ventas','margen','nps','devolucion','clientes','costo_adq','churn','ltv']
n_e2=250
# Estructura: PC1=rentabilidad, PC2=satisfaccion, PC3=retencion
Z_e2=np.random.randn(n_e2,3)
W_e2=np.array([[2.,0.,0.],[1.5,0.,0.],[0.,2.,0.],[-0.3,1.5,0.],
               [1.,0.,0.],[-.5,0.,1.5],[0.,-1.,2.],[0.,0.5,1.8]])
X_e2=Z_e2@W_e2.T+np.random.randn(n_e2,8)*0.3
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED); feats_e2=['ventas','margen','nps','devolucion','clientes','costo_adq','churn','ltv']
n_e2=250; Z_e2=np.random.randn(n_e2,3)
W_e2=np.array([[2.,0.,0.],[1.5,0.,0.],[0.,2.,0.],[-0.3,1.5,0.],[1.,0.,0.],[-.5,0.,1.5],[0.,-1.,2.],[0.,0.5,1.8]])
X_e2=Z_e2@W_e2.T+np.random.randn(n_e2,8)*0.3
pca_e2=PCA(n_components=3); pca_e2.fit(StandardScaler().fit_transform(X_e2))
df_load=pd.DataFrame(pca_e2.components_.T,index=feats_e2,columns=['PC1','PC2','PC3'])
print('Loadings:'); print(df_load.round(3).to_string())
fig,ax=plt.subplots(figsize=(8,4))
im=ax.imshow(df_load.values.T,cmap='RdBu_r',vmin=-1,vmax=1,aspect='auto')
plt.colorbar(im,ax=ax)
ax.set_xticks(range(8)); ax.set_xticklabels(feats_e2,rotation=35,ha='right')
ax.set_yticks(range(3)); ax.set_yticklabels(['PC1','PC2','PC3'])
for i in range(3):
    for j in range(8):
        v=df_load.values[j,i]
        ax.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=8,color='white' if abs(v)>0.5 else 'black')
ax.set_title('Heatmap loadings',fontweight='bold'); plt.tight_layout(); plt.show()
print('\nInterpretación:')
for pc in ['PC1','PC2','PC3']:
    top=df_load[pc].abs().nlargest(2)
    print(f'  {pc}: {list(top.index)} → sugiere "{"Rentabilidad" if pc=="PC1" else "Satisfaccion" if pc=="PC2" else "Retencion"}"')

---
## 🟡 Ejercicio 3 — Error de reconstrucción y elección de k

1. Calcular MSE de reconstrucción para k = 1..p
2. Graficar MSE vs k
3. Identificar el k donde el error se aplana
4. Comparar con el k recomendado por Kaiser
5. ¿Qué porcentaje de información se pierde con el k elegido?

In [ ]:
np.random.seed(SEED)
n_e3, p_e3 = 250, 9
# Señal fuerte en 4 dims, ruido en las últimas
X_e3 = (np.random.randn(n_e3,4)@np.diag([3.,2.5,2.,1.5])@np.random.randn(4,p_e3)
        + np.random.randn(n_e3,p_e3)*0.2)
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED); n_e3,p_e3=250,9
X_e3=np.random.randn(n_e3,4)@np.diag([3.,2.5,2.,1.5])@np.random.randn(4,p_e3)+np.random.randn(n_e3,p_e3)*0.2
X_e3_sc=StandardScaler().fit_transform(X_e3)
pca_all3=PCA(); pca_all3.fit(X_e3_sc)
ev_all3=pca_all3.explained_variance_; k_kai3=int(np.sum(ev_all3>1))
errores3=[]
for k in range(1,p_e3+1):
    p=PCA(n_components=k); Xr=p.fit_transform(X_e3_sc); errores3.append(np.mean((X_e3_sc-p.inverse_transform(Xr))**2))
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(range(1,p_e3+1),errores3,'o-',color='#7D3C98',lw=2,markersize=8)
ax.axvline(k_kai3,color='#E74C3C',lw=2,linestyle='--',label=f'Kaiser k={k_kai3}')
ax.set(xlabel='k PCs',ylabel='MSE reconstrucción',title='Error de reconstrucción')
ax.legend(); ax.grid(True,alpha=0.25); plt.tight_layout(); plt.show()
print(f'Kaiser: k={k_kai3}  Error={errores3[k_kai3-1]:.4f}  Info retenida: {(1-errores3[k_kai3-1])*100:.1f}%')

---
## 🔴 Ejercicio 4 — Pipeline PCA + clasificación con selección de k

Implementar `pca_pipeline_search(X, y, ks)` que:
1. Prueba PCA + Logística para cada k en ks
2. Evalúa accuracy en test
3. Reporta tabla varianza vs accuracy
4. Elige el k con mejor accuracy
5. Grafica accuracy vs k y marca el óptimo

In [ ]:
def pca_pipeline_search(X, y, ks=range(1, 11)):
    pass

np.random.seed(SEED)
n_e4, p_e4 = 500, 15
# Señal en 5 dimensiones latentes
Z_e4=np.random.randn(n_e4, 5)
X_e4=Z_e4@np.random.randn(5, p_e4)+np.random.randn(n_e4, p_e4)*0.5
y_e4=np.random.binomial(1, 1/(1+np.exp(-(Z_e4@np.array([1.5,-1.2,0.8,-0.6,1.0])-0.3))))
pca_pipeline_search(X_e4, y_e4, ks=range(1, 13))

In [ ]:
# ✅ SOLUCIÓN
def pca_pipeline_search(X,y,ks=range(1,11)):
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,random_state=SEED)
    results=[]
    print(f'{"k":>4s} {"Var%":>8s} {"Accuracy":>10s}')
    print('─'*26)
    for k in ks:
        pipe=Pipeline([('sc',StandardScaler()),('pca',PCA(n_components=k)),
                       ('lr',LogisticRegression(C=1e6,solver='lbfgs',max_iter=500))])
        pipe.fit(Xtr,ytr); acc=accuracy_score(yte,pipe.predict(Xte))
        var=pipe['pca'].explained_variance_ratio_.sum()
        results.append((k,var,acc))
        print(f'{k:>4d} {var:>8.1%} {acc:>10.4f}')
    best=max(results,key=lambda x:x[2])
    print(f'\nMejor k={best[0]}  Var={best[1]:.1%}  Acc={best[2]:.4f}')
    ks_r,vars_r,accs_r=zip(*results)
    fig,ax=plt.subplots(figsize=(9,4))
    ax.plot(ks_r,accs_r,'o-',color='#7D3C98',lw=2,markersize=8)
    ax.axvline(best[0],color='#E74C3C',lw=2,linestyle='--',label=f'Mejor k={best[0]}')
    ax.set(xlabel='k PCs',ylabel='Accuracy',title='Accuracy vs número de componentes')
    ax.legend(); ax.grid(True,alpha=0.25); plt.tight_layout(); plt.show()
    return best

np.random.seed(SEED); n_e4,p_e4=500,15
Z_e4=np.random.randn(n_e4,5)
X_e4=Z_e4@np.random.randn(5,p_e4)+np.random.randn(n_e4,p_e4)*0.5
y_e4=np.random.binomial(1,1/(1+np.exp(-(Z_e4@np.array([1.5,-1.2,0.8,-0.6,1.0])-0.3))))
pca_pipeline_search(X_e4,y_e4,ks=range(1,13))

---
<div style="background:#512E5F;color:white;padding:28px 24px;border-radius:8px;">

<strong>🎓 Módulo 4 y Curso 8 completados</strong>

PCA · SVD · Scree plot · Biplot · Kaiser · Codo · Loadings · Pipeline PCA+clasificación

**Josef Rodriguez · Curso 8 · Modelos Estadísticos**

</div>